# DLinear 후속 실험: Table 8 시간·메모리 + 논문 당시 코드로 재실행
`DLinear_extra_experiments.ipynb` 다음에 돌리는 노트북.

| 순서 | 실험 | 확인하는 것 | 예상 시간 (T4) |
|---|---|---|---|
| 1 | Table 8 효율 | Electricity, L=96, T=720에서 추론 시간과 GPU 메모리 | 1분 |
| 2 | 논문 당시 코드 | 2022-06 커밋 `04ffd8d`의 코드·스크립트 그대로 Exchange · ILI · Weather 12칸 (+ ILI 학습률 변형 4칸) | 15분 |
| 3 | (선택) 같은 방식으로 Electricity · Traffic 8칸 | **High-RAM 필요** | 2~3시간 |

**실행:** 서버는 **GPU → T4** (3번까지 돌릴 거면 **High-RAM**) → **Run All**. 3번은 맨 아래 셀이라 필요 없으면 중간에 멈춰도 된다.

**실험 2의 배경.** 지금 공식 코드는 논문 표가 만들어진 2022년 6월 코드와 세 가지가 다르다.
- 모델: 선형층 가중치를 $1/L$로 채우고, 쓰지 않는 선형층(`Linear_Decoder`)을 하나 더 만들었다 (난수 소비 순서가 달라짐). 2022-08 `7f7a25b`에서 삭제.
- 스크립트: 데이터셋마다 학습률·배치 크기가 달랐다 (예: Exchange 720 lr 0.005, ILI lr 0.05, ETTh1 batch 8). 2022-08 `7f7a25b`에서 변경.
- 평가: test `drop_last=True` (마지막 덜 찬 배치를 버림). 2023-03 `6fe4c28`에서 False로 변경. 메인테이너가 issue #58에서 확인.

ETT 4개 데이터셋(다변량 16칸 + 단변량 16칸)은 로컬에서 이 코드로 이미 돌려 봤다 (`results/paper_era_ett.csv`). 다변량 16칸은 전부 논문과 소수 셋째 자리까지 같았다 (ETTm2 720만 2022-08-22에 고쳐진 학습률 0.01 적용). 여기서는 Google Drive 데이터가 필요한 나머지 데이터셋을 같은 방식으로 돌린다.

In [ ]:
# 0. GPU 확인
!nvidia-smi --query-gpu=name,memory.total --format=csv
!free -g | head -2

In [ ]:
# 1. 코드·데이터 (이미 있으면 건너뜀)
import os, subprocess
os.chdir('/content')
if not os.path.exists('LTSF-Linear'):
    subprocess.run(['git', 'clone', '-q', 'https://github.com/cure-lab/LTSF-Linear.git'], check=True)
os.chdir('/content/LTSF-Linear')

def patch(path, old, new):
    s = open(path).read()
    if new in s:
        return
    assert old in s, (path, old)
    open(path, 'w').write(s.replace(old, new))

os.makedirs('dataset', exist_ok=True)
need = ['electricity', 'exchange_rate', 'national_illness', 'traffic', 'weather']
if not all(os.path.exists(f'dataset/{f}.csv') for f in need):
    subprocess.run('pip -q install gdown && gdown -q --folder '
                   'https://drive.google.com/drive/folders/1ZOYpTUa82_jCcxIdTmyr0LXQfvaM9vIy -O raw_data && '
                   'find raw_data -name "*.csv" -exec cp -n {} dataset/ \\;', shell=True, check=True)

# 논문 당시 코드(2022-06-13, 04ffd8d)를 옆 폴더에 꺼낸다. 지금 폴더는 그대로 둔다.
OLD = '/content/LTSF-old'
if not os.path.exists(OLD):
    r = subprocess.run(['git', 'worktree', 'add', '-q', OLD, '04ffd8d'])
    if r.returncode != 0:                         # 얕은 clone이면 이력을 받아서 다시
        subprocess.run(['git', 'fetch', '-q', '--unshallow'])
        subprocess.run(['git', 'worktree', 'add', '-q', OLD, '04ffd8d'], check=True)
patch(f'{OLD}/utils/tools.py', 'np.Inf', 'np.inf')   # NumPy 2 호환 (이 한 줄만 고침)
os.makedirs(f'{OLD}/dataset', exist_ok=True)
for f in os.listdir('dataset'):
    if not os.path.exists(f'{OLD}/dataset/{f}'):
        os.symlink(f'/content/LTSF-Linear/dataset/{f}', f'{OLD}/dataset/{f}')
!git -C /content/LTSF-old log --oneline -1
!grep -n "drop_last = True" /content/LTSF-old/data_provider/data_factory.py
!grep -n "Linear_Decoder = nn.Linear\|1/self.seq_len" /content/LTSF-old/models/DLinear.py
!ls -lh /content/LTSF-Linear/dataset

In [ ]:
%%writefile /content/lab.py
"""DLinear 추가 실험 도우미.

공식 스크립트(scripts/EXP-*/...sh)를 '가짜 python'으로 한 번 돌려서 실제 실행될 명령 목록만 뽑고,
그 명령을 하나씩 실행한다. 이렇게 하면
  - 하이퍼파라미터를 손으로 옮겨 적지 않고 공식 스크립트 값을 그대로 쓰고
  - 실험마다 진행 상황이 한 줄씩 찍히고
  - 중간에 끊겨도 다시 실행하면 끝난 실험은 건너뛴다.
"""
import json, os, re, shutil, subprocess, sys, time

import pandas as pd

ROOT = '/content/LTSF-Linear'
_SHIM = '/content/_shim'
_WORK = '/content/_collect'

DATASETS = {'etth1.csv': 'ETTh1', 'etth2.csv': 'ETTh2', 'ettm1.csv': 'ETTm1', 'ettm2.csv': 'ETTm2',
            'electricity.csv': 'Electricity', 'exchange_rate.csv': 'Exchange', 'traffic.csv': 'Traffic',
            'weather.csv': 'Weather', 'national_illness.csv': 'ILI'}


def commands(script, edits=()):
    """공식 스크립트에서 실행될 python 명령 목록을 뽑는다. edits = [(찾을 문자열, 바꿀 문자열), ...]"""
    os.makedirs(_SHIM, exist_ok=True)
    with open(f'{_SHIM}/python', 'w') as f:
        f.write('#!/usr/bin/env python3\n'
                'import json, os, sys\n'
                'try:\n    log = os.readlink("/proc/self/fd/1")\nexcept OSError:\n    log = ""\n'
                'with open(os.environ["CMD_FILE"], "a") as f:\n'
                '    f.write(json.dumps({"argv": sys.argv[1:], "log": log}) + "\\n")\n')
    os.chmod(f'{_SHIM}/python', 0o755)

    shutil.rmtree(_WORK, ignore_errors=True)
    for d in ['logs/LongForecasting/univariate', 'logs/LookBackWindow']:  # 스크립트가 안 만드는 폴더도 미리
        os.makedirs(f'{_WORK}/{d}')
    src = open(os.path.join(ROOT, script)).read()
    for a, b in edits:
        assert a in src, f'{a!r} 가 {script} 에 없음'
        src = src.replace(a, b)
    open(f'{_WORK}/run.sh', 'w').write(src)
    env = dict(os.environ, PATH=f'{_SHIM}:{os.environ["PATH"]}', CMD_FILE=f'{_WORK}/cmds.jsonl')
    subprocess.run(['bash', 'run.sh'], cwd=_WORK, env=env, check=True, stdout=subprocess.DEVNULL)
    out = [json.loads(l) for l in open(f'{_WORK}/cmds.jsonl')]
    for c in out:
        c['log'] = os.path.basename(c['log']) if c['log'].startswith(_WORK) else ''
    return out


def arg(c, *names, default=None):
    a = c['argv']
    for i, x in enumerate(a[:-1]):
        if x in names:
            return a[i + 1]
    return default


def set_arg(c, name, value):
    a = c['argv']
    if name in a:
        a[a.index(name) + 1] = str(value)
    else:
        a += [name, str(value)]
    return c


def info(c):
    return dict(model=arg(c, '--model'), dataset=DATASETS[arg(c, '--data_path').lower()],
                features=arg(c, '--features', '--feature', default='M'),
                seq_len=int(arg(c, '--seq_len')), pred_len=int(arg(c, '--pred_len')),
                individual='--individual' in c['argv'])


def parse_log(path):
    """로그에서 (testing 설정, mse, mae)를 순서대로 뽑는다. --itr N이면 N개."""
    if not os.path.exists(path):
        return []
    text = open(path, errors='ignore').read()
    out = []
    for m in re.finditer(r'>>>>>>>testing : (\S+?)<<<.*?mse:([-\d.eE+]+), mae:([-\d.eE+]+)', text, re.S):
        out.append((m.group(1), float(m.group(2)), float(m.group(3))))
    return out


def run(cmds, tag):
    """명령을 차례로 실행. 로그는 logs/<tag>/ 에 저장, 이미 끝난 실험은 건너뜀."""
    logdir = os.path.join(ROOT, 'logs', tag)
    os.makedirs(logdir, exist_ok=True)
    t0 = time.time()
    for i, c in enumerate(cmds, 1):
        c['logpath'] = os.path.join(logdir, c.get('name') or c['log'] or f'run{i}.log')
        d = info(c)
        label = (f"{d['dataset']} {d['features']} L={d['seq_len']} T={d['pred_len']}"
                 + (' individual' if d['individual'] else ''))
        need = int(arg(c, '--itr', default=1))
        if len(parse_log(c['logpath'])) >= need:
            print(f'[{i}/{len(cmds)}] {label}: 이미 완료, 건너뜀', flush=True)
            continue
        t = time.time()
        with open(c['logpath'], 'w') as f:
            r = subprocess.run([sys.executable] + c['argv'], cwd=ROOT, stdout=f, stderr=subprocess.STDOUT)
        subprocess.run('rm -f results/*/pred.npy', shell=True, cwd=ROOT)  # 예측 배열은 안 쓰므로 디스크 절약
        res = parse_log(c['logpath'])
        if r.returncode != 0 or len(res) < need:
            tail = open(c['logpath'], errors='ignore').read()[-600:]
            print(f'[{i}/{len(cmds)}] {label}: 실패 (exit {r.returncode})\n{tail}', flush=True)
            continue
        mses = ', '.join(f'{m:.4f}' for _, m, _ in res)
        print(f'[{i}/{len(cmds)}] {label}: mse {mses}  ({time.time() - t:.0f}s, 누적 {(time.time() - t0) / 60:.0f}분)',
              flush=True)


def collect(cmds):
    rows = []
    for c in cmds:
        for k, (setting, mse, mae) in enumerate(parse_log(c.get('logpath', ''))):
            rows.append(dict(info(c), run=k, mse=mse, mae=mae))
    return pd.DataFrame(rows)

## 실험 1. 논문 Table 8 DLinear 행 (Electricity, L=96, T=720)
논문 값: MACs 0.04G, 파라미터 139.7K, 시간 0.4ms, 메모리 687MiB (Titan XP, 시간은 5회 평균).
논문은 배치 크기와 메모리 측정 방식을 적지 않아서 batch 1·16·32를 모두 잰다. 메모리는 텐서가 실제로 쓴 양(peak)과 CUDA context를 포함한 GPU 전체 사용량(nvidia-smi) 두 가지를 본다.

In [ ]:
# 논문 Table 8 DLinear 행: Electricity, L=96, T=720 (변수 321개)
import os, subprocess, sys, time, types, statistics
if not os.path.isdir('/content/LTSF-Linear'):
    subprocess.run(['git', 'clone', '-q', 'https://github.com/cure-lab/LTSF-Linear.git', '/content/LTSF-Linear'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'ptflops'], check=True)
sys.path.insert(0, '/content/LTSF-Linear')
import torch
from models import DLinear
from ptflops import get_model_complexity_info

dev = 'cuda' if torch.cuda.is_available() else 'cpu'
args = types.SimpleNamespace(seq_len=96, pred_len=720, individual=False, enc_in=321)
model = DLinear.Model(args).to(dev).eval()

n_params = sum(p.numel() for p in model.parameters())
macs, _ = get_model_complexity_info(model, (96, 321), as_strings=False,
                                    print_per_layer_stat=False, verbose=False)
print(f'device: {torch.cuda.get_device_name(0) if dev == "cuda" else "cpu"}')
print(f'params: {n_params:,} ({n_params/1e3:.1f}K)   MACs (batch 1): {macs/1e9:.3f}G')

def sync():
    if dev == 'cuda':
        torch.cuda.synchronize()

for bs in [1, 16, 32]:
    x = torch.randn(bs, 96, 321, device=dev)
    if dev == 'cuda':
        torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()
    with torch.no_grad():
        for _ in range(50):          # warm-up
            model(x)
        sync()
        runs = []
        for _ in range(5):           # 논문: 5회 평균
            t0 = time.perf_counter()
            for _ in range(200):
                model(x)
            sync()
            runs.append((time.perf_counter() - t0) / 200 * 1e3)
    line = f'batch {bs:>2}: {statistics.mean(runs):.3f} ms/step (5 runs: ' + ', '.join(f'{r:.3f}' for r in runs) + ')'
    if dev == 'cuda':
        line += f'   peak tensor memory: {torch.cuda.max_memory_allocated()/2**20:.1f} MiB'
    print(line)

if dev == 'cuda':
    used = subprocess.run(['nvidia-smi', '--query-gpu=memory.used', '--format=csv,noheader'],
                          capture_output=True, text=True).stdout.strip()
    print(f'nvidia-smi memory.used (CUDA context 포함): {used}')

## 실험 2. 논문 당시 코드로 Exchange · ILI · Weather
2022-06 코드와 그때의 `scripts/EXP-LongForecasting/DLinear/*.sh`를 고치지 않고 그대로 돌린다. seed 2021, 1회 (논문과 같은 조건, issue #8에서 메인테이너가 "모든 결과는 seed 2021"이라고 답함).

In [ ]:
import sys, importlib
import numpy as np, pandas as pd
sys.path.insert(0, '/content'); import lab; importlib.reload(lab)
lab.ROOT = OLD                                        # lab.run이 옛 코드 폴더에서 실행하도록
pd.set_option('display.width', 200)
SCR = f'{OLD}/scripts/EXP-LongForecasting/DLinear/'

PAPER = {  # 논문 Table 2 DLinear (MSE, MAE)
 'Exchange': {96: (0.081, 0.203), 192: (0.157, 0.293), 336: (0.305, 0.414), 720: (0.643, 0.601)},
 'Weather': {96: (0.176, 0.237), 192: (0.220, 0.282), 336: (0.265, 0.319), 720: (0.323, 0.362)},
 'Electricity': {96: (0.140, 0.237), 192: (0.153, 0.249), 336: (0.169, 0.267), 720: (0.203, 0.301)},
 'Traffic': {96: (0.410, 0.282), 192: (0.423, 0.287), 336: (0.436, 0.296), 720: (0.466, 0.315)},
 'ILI': {24: (2.215, 1.081), 36: (1.963, 0.963), 48: (2.130, 1.024), 60: (2.368, 1.096)},
}
NOW = {  # 지금 공식 코드로 한 첫 재현 (MSE)
 'Exchange': {96: 0.085, 192: 0.162, 336: 0.333, 720: 0.898}, 'Weather': {96: 0.174, 192: 0.218, 336: 0.263, 720: 0.332},
 'Electricity': {96: 0.140, 192: 0.154, 336: 0.169, 720: 0.204}, 'Traffic': {96: 0.413, 192: 0.424, 336: 0.438, 720: 0.466},
 'ILI': {24: 2.280, 36: 2.235, 48: 2.298, 60: 2.573},
}

def old_cmds(names):
    out = []
    for n in names:
        for c in lab.commands(SCR + n + '.sh'):
            lab.set_arg(c, '--num_workers', 2)        # Colab CPU 수에 맞춤 (결과 영향 없음)
            out.append(c)
    return out

def table(cmds):
    rows = []
    for c in cmds:
        d = lab.info(c)
        for _, mse, mae in lab.parse_log(c['logpath']):
            pm, pa = PAPER[d['dataset']][d['pred_len']]
            rows.append(dict(dataset=d['dataset'], pred_len=d['pred_len'], lr=lab.arg(c, '--learning_rate'),
                             batch=lab.arg(c, '--batch_size'), mse_old=mse, mae_old=mae, paper_mse=pm, paper_mae=pa,
                             mse_now=NOW[d['dataset']][d['pred_len']]))
    df = pd.DataFrame(rows)
    df['old-paper'] = df['mse_old'] - df['paper_mse']
    df['now-paper'] = df['mse_now'] - df['paper_mse']
    return df

EXP2 = old_cmds(['exchange_rate', 'ili', 'weather'])
lab.run(EXP2, 'paper_era')
R2 = table(EXP2)
R2['setting'] = '2022-06 코드·스크립트'
R2.round(4)

### 실험 2-2. ILI만 한 번 더
2022-06 README 표의 ILI 값(2.108 / 2.332 / 2.088 / 2.299)은 논문 Table 2(2.215 / 1.963 / 2.130 / 2.368)와 달라서, 논문의 ILI는 나중에 다시 돌린 값으로 보인다.
2022-08에 ILI 학습률이 0.05 → 0.01로 바뀌었으므로, 옛 코드에 학습률만 0.01로 바꿔서도 돌려 본다.

In [ ]:
EXP2B = old_cmds(['ili'])
for c in EXP2B:
    lab.set_arg(c, '--learning_rate', '0.01')
    lab.set_arg(c, '--model_id', lab.arg(c, '--model_id') + '_lr001')
lab.run(EXP2B, 'paper_era_ili_lr001')
R2B = table(EXP2B)
R2B['setting'] = '2022-06 코드 + ILI lr 0.01'
R2 = pd.concat([R2, R2B], ignore_index=True)
R2B.round(4)

## 3. 결과 복사용
아래 출력과 실험 1 출력을 복사해서 붙여 주면 정리한다.

In [ ]:
print(R2.to_csv(index=False, float_format='%.4f'))
os.makedirs('/content/extra_results', exist_ok=True)
R2.to_csv('/content/extra_results/paper_era_small.csv', index=False)

## 실험 3. (선택) Electricity · Traffic — High-RAM 필요, 2~3시간
첫 재현에서도 거의 맞았던 데이터셋이라 꼭 돌릴 필요는 없다. 옛 코드는 test 입력 배열까지 메모리에 모으므로 **High-RAM 런타임**에서만 돌린다.

In [ ]:
EXP3 = old_cmds(['electricity', 'traffic'])
lab.run(EXP3, 'paper_era_big')
R3 = table(EXP3)
print(R3.to_csv(index=False, float_format='%.4f'))
R3.round(4)